# 04 · Search strategies compared

Replays three ways of autotuning over the measured sweep: trying every config, trying k random
configs (20 seeds), and trying ArchTune's top k. A strategy is charged the compile and benchmark
time of each config it tries. Training and held-out shapes are reported separately.

The last cell repeats the comparison live with `triton.autotune` on three held-out shapes.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = False

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
import os, pandas as pd
GPU = ENV["gpu_tag"] or "rtx3050_sm86"   # results folder to analyse
if not os.path.exists(os.path.join(RESULTS, GPU, "analysis", "curves.parquet")):
    run.analyze(RESULTS, GPU)
curves = pd.read_parquet(os.path.join(RESULTS, GPU, "analysis", "curves.parquet"))
run.fig1_quality_vs_cost(curves, GPU, os.path.join(RESULTS, "figures", f"fig1_quality_vs_cost_{GPU}.png"))

In [ ]:
table6 = pd.read_csv(os.path.join(RESULTS, GPU, "analysis", "table6.csv"))
display(table6)
display(model.cost_ratio(table6))

## Live check on held-out shapes

In [ ]:
import json, torch
if torch.cuda.is_available() and gpu.pinned_ok(ENV):
    reg, smem = run.load_models(RESULTS, GPU)
    A = gpu.live()
    sw = model.prepare(run.load_sweep(RESULTS, GPU))
    split = json.load(open(os.path.join(RESULTS, GPU, "analysis", "summary.json")))["split"]
    rows = []
    for sid in [s for s, v in split.items() if v == "test" and s.startswith("mm_")][:3]:
        r0 = sw[sw.shape_id == sid].iloc[0]
        cfgs = model.mini_matmul_space() if MINI else model.matmul_space()
        live = model.tune_live(cfgs, int(r0.M), int(r0.N), int(r0.K),
                               prune_by=model.archtune_prune_configs_by(A, config.EXPERIMENTS.spotcheck_k, reg, smem))
        best = model.best_time(sw[sw.shape_id == sid])
        rows.append({**live, "shape_id": sid, "sweep_best_ms": best, "quality": best / live["time_ms_median"]})
    spot = pd.DataFrame(rows); spot.to_parquet(os.path.join(RESULTS, GPU, "spotcheck.parquet")); display(spot)
else:
    print("needs a GPU with the pinned torch/triton")